# CSV Basics — 10: Streaming CSV with Structured Streaming

## What you will learn
Structured Streaming can process CSV files as they arrive in a directory.
This is useful for legacy systems that export CSV files on a schedule.

In this notebook:
1. File-based streaming source for CSV
2. Schema enforcement in streaming
3. `maxFilesPerTrigger` and processing rate control
4. Error handling in streaming CSV
5. Archiving processed files
6. Monitoring streaming progress


In [ ]:
import os, time, pathlib, shutil
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/csv_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('csv-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version} | DATA_DIR: {DATA_DIR}")

In [ ]:
import pathlib, shutil, threading, time, random, datetime, json as pyjson

# Setup directories
STREAM_INPUT  = f"{DATA_DIR}/stream_csv_input"
STREAM_OUTPUT = f"{DATA_DIR}/stream_csv_output"
STREAM_CKPT   = f"{DATA_DIR}/stream_csv_checkpoint"
STREAM_ARCHIVE= f"{DATA_DIR}/stream_csv_archive"

for d in [STREAM_INPUT, STREAM_OUTPUT, STREAM_CKPT, STREAM_ARCHIVE]:
    shutil.rmtree(d, ignore_errors=True)
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)

print("Streaming directories ready:")
print(f"  Input    : {STREAM_INPUT}   (drop CSV files here)")
print(f"  Output   : {STREAM_OUTPUT}  (processed Parquet)")
print(f"  Checkpoint: {STREAM_CKPT}")

## Step 1 — Define Schema and Streaming Reader

In streaming, schema MUST be defined explicitly.
`inferSchema` is not supported for streaming DataFrames.


In [ ]:
event_schema = StructType([
    StructField("event_id",   LongType()),
    StructField("user_id",    LongType()),
    StructField("event_type", StringType()),
    StructField("page",       StringType()),
    StructField("revenue",    DoubleType()),
    StructField("event_date", DateType()),
])

# Streaming reader — watches STREAM_INPUT for new CSV files
csv_stream = (
    spark.readStream
         .format("csv")
         .schema(event_schema)
         .option("header", True)
         .option("maxFilesPerTrigger", 1)     # process 1 file per micro-batch
         .option("ignoreLeadingWhiteSpace", True)
         .option("ignoreTrailingWhiteSpace", True)
         .option("mode", "PERMISSIVE")        # don't fail on bad rows
         .option("columnNameOfCorruptRecord", "_corrupt_record")
         .load(STREAM_INPUT)
)

print(f"Is streaming: {csv_stream.isStreaming}")
print("Schema:")
csv_stream.printSchema()

## Step 2 — Write Stream to Parquet Output


In [ ]:
# Process: filter good rows, add processing timestamp
processed = (
    csv_stream
    .filter(F.col("_corrupt_record").isNull())  # only good rows
    .drop("_corrupt_record")
    .withColumn("_processed_at", F.current_timestamp())
    .withColumn("_source_date",  F.current_date())
)

# Write to Parquet (partitioned by source date)
query = (
    processed
    .writeStream
    .format("parquet")
    .outputMode("append")
    .option("checkpointLocation", STREAM_CKPT)
    .option("path", STREAM_OUTPUT)
    .partitionBy("_source_date")
    .queryName("csv_stream_processor")
    .start()
)

print(f"Streaming query started: {query.name}")
print(f"Status: {query.status['message']}")

## Step 3 — Generate CSV Files to Stream


In [ ]:
EVENTS    = ["page_view","click","purchase","search","add_to_cart"]
PAGES     = ["/home","/products","/cart","/checkout"]

def generate_csv_batch(batch_id, n=50):
    """Write a CSV file to the streaming input directory."""
    base = datetime.date(2024, 1, 1)
    rows = ["event_id,user_id,event_type,page,revenue,event_date"]
    for i in range(n):
        d = base + datetime.timedelta(days=random.randint(0, 90))
        rev = round(random.uniform(10, 500), 2) if random.random() > 0.7 else 0.0
        rows.append(
            f"{batch_id*1000+i},{random.randint(1,500)},"
            f"{random.choice(EVENTS)},{random.choice(PAGES)},"
            f"{rev},{d}"
        )
    path = f"{STREAM_INPUT}/events_{batch_id:04d}.csv"
    pathlib.Path(path).write_text("\n".join(rows))
    return path

print("Generating CSV batches for streaming...")
for i in range(1, 5):
    path = generate_csv_batch(i)
    print(f"  Dropped: {path.split('/')[-1]}")
    time.sleep(3)

time.sleep(5)
print("\nStream processed. Checking output...")

if pathlib.Path(STREAM_OUTPUT).exists():
    output_count = spark.read.parquet(STREAM_OUTPUT).count()
    print(f"  Output rows: {output_count:,}")
else:
    print("  Output not yet written")

## Step 4 — Monitor Streaming Progress


In [ ]:
# Check last progress
if query.lastProgress:
    prog = query.lastProgress
    print("Last batch progress:")
    print(f"  Batch ID           : {prog.get('batchId')}")
    print(f"  Input rows         : {prog.get('numInputRows')}")
    print(f"  Rows/sec           : {prog.get('processedRowsPerSecond', 0):.1f}")
    print(f"  Trigger duration   : {prog.get('durationMs', {}).get('triggerExecution', 0)} ms")
    print()
    print(f"  Source (files seen) : {prog.get('sources', [{}])[0].get('numInputRows', 0)}")

print("\nAll progress batches:")
for p in query.recentProgress[-3:]:
    print(f"  batch={p.get('batchId')} rows={p.get('numInputRows')} "
          f"rate={p.get('processedRowsPerSecond',0):.0f}/s")

query.stop()
print("\nStream stopped.")

## Step 5 — Final Output Verification


In [ ]:
# Verify output
print("Streaming pipeline output:")
try:
    df_output = spark.read.parquet(STREAM_OUTPUT)
    print(f"  Total rows processed: {df_output.count():,}")
    print()
    print("  Distribution by event type:")
    df_output.groupBy("event_type").count().orderBy(F.desc("count")).show()
    print("  Revenue summary:")
    df_output.filter(F.col("revenue") > 0) \
             .agg(F.count("*").alias("purchase_events"),
                  F.sum("revenue").alias("total_revenue"),
                  F.avg("revenue").alias("avg_revenue")).show()
except Exception as e:
    print(f"  No output yet: {e}")

## Summary: Streaming CSV Pattern

```python
# Define schema ALWAYS (inferSchema not supported in streaming)
schema = StructType([...])

# Streaming reader
stream = (
    spark.readStream
         .format("csv")
         .schema(schema)
         .option("header", True)
         .option("maxFilesPerTrigger", 1)     # rate control
         .option("mode", "PERMISSIVE")
         .load("/path/to/input/dir")
)

# Write to Parquet
query = (
    stream.writeStream
          .format("parquet")
          .outputMode("append")
          .option("checkpointLocation", "/path/checkpoint")
          .option("path", "/path/output")
          .start()
)
```

### maxFilesPerTrigger tuning
- `1` — conservative, low latency, high trigger overhead
- `5–10` — balanced for most cases
- `1000` — batch-like, process large backlog quickly

### Production considerations
- Always use `columnNameOfCorruptRecord` for bad row capture
- Monitor `numInputRows` — if 0 for many batches, check input path
- Checkpoint enables exactly-once — never delete it while stream is running
